# Event Study Analysis
## Section 232 Tariff Impact Study

This notebook implements an **event-study specification** to examine dynamic treatment effects.

**Key Questions:**
1. How did the tariff impact evolve over time?
2. Were there anticipation effects before implementation?
3. Did effects persist or fade over time?

### Specification:
```
Y_ict = α + Σ β_τ·(Treated_i × 1[t=τ]) + γ_ic + δ_t + ε_ict
```

Where τ indexes time relative to treatment (τ = -12, ..., -1, 0, 1, ..., +24)

In [ ]:
import sys
sys.path.append('../models')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from datetime import datetime
from did_model import DIDModel

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

%matplotlib inline
%load_ext autoreload
%autoreload 2

## 1. Load Data and Setup

In [ ]:
# Load panel data
data_path = '../data/clean/steel_panel.parquet'
df = pd.read_parquet(data_path)

# Treatment configuration
TREATMENT_DATE = pd.to_datetime('2018-03-23')
EXEMPT_COUNTRIES = ['Canada', 'Mexico']

print(f"Loaded {len(df):,} observations")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
df.head()

## 2. Initialize DID Model and Prepare Data

In [ ]:
# Initialize DID model
model = DIDModel(treatment_date='2018-03-23')

# Load data
model.data = df

# Create treatment indicators
treated_countries = [c for c in df['country'].unique() if c not in EXEMPT_COUNTRIES]
model.create_treatment_indicators(treated_countries=treated_countries)

print(f"\nTreated countries: {len(treated_countries)}")
print(f"Control countries: {len(EXEMPT_COUNTRIES)}")

## 3. Fit Event-Study Model

Estimate separate coefficients for each month relative to tariff implementation.

In [ ]:
# Fit event study with leads and lags
results = model.fit_event_study(
    outcome_var='log_import_value',
    leads=12,   # 12 months pre-treatment
    lags=24,    # 24 months post-treatment
    cluster_var='country'
)

print("\nEvent-Study Model Summary:")
print(f"R-squared: {results.rsquared:.4f}")
print(f"Observations: {results.nobs:,.0f}")
print(f"Clusters: {model.data['country'].nunique()}")

## 4. Extract and Examine Coefficients

In [ ]:
# Extract event-time coefficients
coefs = model.results['event_study_coefs']
ses = model.results['event_study_ses']

# Create DataFrame for analysis
event_df = pd.DataFrame({
    'rel_month': list(coefs.keys()),
    'coefficient': list(coefs.values()),
    'std_error': list(ses.values())
})

event_df['ci_lower'] = event_df['coefficient'] - 1.96 * event_df['std_error']
event_df['ci_upper'] = event_df['coefficient'] + 1.96 * event_df['std_error']
event_df['significant'] = (event_df['ci_lower'] > 0) | (event_df['ci_upper'] < 0)

# Display coefficients
print("\nEvent-Time Coefficients:")
print(event_df.to_string(index=False))

## 5. Plot Event-Study Coefficients

In [ ]:
# Generate plot using model method
fig = model.plot_event_study(
    title='Event Study: Impact of Section 232 Tariffs on Steel Imports',
    save_path='../viz/event_study_main.png'
)

plt.show()

## 6. Test for Pre-Treatment Effects

Test whether pre-treatment coefficients are jointly zero (no anticipation effects).

In [ ]:
# Extract pre-treatment coefficients (τ < 0, excluding reference period τ = -1)
pre_coefs = event_df[event_df['rel_month'] < -1]['coefficient'].values

# Simple test: Are they all close to zero?
mean_pre = np.mean(pre_coefs)
std_pre = np.std(pre_coefs)
max_abs_pre = np.max(np.abs(pre_coefs))

print("\n" + "="*60)
print("PRE-TREATMENT EFFECTS TEST")
print("="*60)
print(f"Mean pre-treatment coefficient:     {mean_pre:.6f}")
print(f"Std. dev. of pre-treatment coefs:   {std_pre:.6f}")
print(f"Max absolute pre-treatment coef:    {max_abs_pre:.6f}")

# Count significant pre-treatment coefficients
n_sig_pre = event_df[(event_df['rel_month'] < -1) & event_df['significant']].shape[0]
n_pre = event_df[event_df['rel_month'] < -1].shape[0]

print(f"\nSignificant pre-treatment coefs:    {n_sig_pre} / {n_pre}")

if n_sig_pre == 0 and max_abs_pre < 0.1:
    print("\n✓ No evidence of pre-treatment effects (anticipation)")
else:
    print("\n⚠️  Some pre-treatment coefficients are significant")
    print("   This may indicate anticipation effects or violations of parallel trends.")

print("="*60)

## 7. Analyze Post-Treatment Dynamics

In [ ]:
# Extract post-treatment coefficients
post_df = event_df[event_df['rel_month'] >= 0].copy()

# Summary statistics
print("\nPost-Treatment Effect Dynamics:")
print(f"  Immediate effect (τ=0):       {post_df[post_df['rel_month']==0]['coefficient'].values[0]:.4f}")
print(f"  Average effect (first 6 mo):  {post_df[post_df['rel_month']<=6]['coefficient'].mean():.4f}")
print(f"  Average effect (7-12 mo):     {post_df[(post_df['rel_month']>6) & (post_df['rel_month']<=12)]['coefficient'].mean():.4f}")
print(f"  Average effect (13-24 mo):    {post_df[post_df['rel_month']>12]['coefficient'].mean():.4f}")

# Plot post-treatment trajectory
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(post_df['rel_month'], post_df['coefficient'], 'o-', linewidth=2.5, markersize=8, color='darkblue')
ax.fill_between(post_df['rel_month'], post_df['ci_lower'], post_df['ci_upper'], 
                alpha=0.2, color='darkblue', label='95% CI')
ax.axhline(0, color='black', linestyle='--', linewidth=1.5, alpha=0.5)

ax.set_xlabel('Months After Tariff Implementation', fontsize=12)
ax.set_ylabel('Coefficient Estimate (Log Points)', fontsize=12)
ax.set_title('Post-Treatment Effect Dynamics', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../viz/post_treatment_dynamics.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. By-Product Event Studies

Examine heterogeneous effects across steel products.

In [ ]:
# Get unique products
products = model.data['product'].unique()

print(f"Running event studies for {len(products)} products...\n")

product_results = {}

for product in products[:3]:  # Limit to first 3 for demonstration
    print(f"Estimating for {product}...")
    
    # Filter data to product
    product_data = model.data[model.data['product'] == product].copy()
    
    # Create relative time variable
    product_data['rel_month'] = ((product_data['date'].dt.year - TREATMENT_DATE.year) * 12 +
                                  (product_data['date'].dt.month - TREATMENT_DATE.month))
    
    # Create log outcome
    product_data['log_value'] = np.log(product_data['import_value'] + 1)
    
    # Create event dummies
    event_vars = []
    for tau in range(-12, 25):
        if tau == -1:
            continue
        dummy_name = f'event_{tau:+d}'
        product_data[dummy_name] = ((product_data['rel_month'] == tau) & 
                                     (product_data['treated'] == 1)).astype(int)
        event_vars.append(dummy_name)
    
    # Run regression
    formula = 'log_value ~ ' + ' + '.join(event_vars) + ' + C(country) + C(date)'
    model_prod = smf.ols(formula, data=product_data)
    results_prod = model_prod.fit(cov_type='cluster', cov_kwds={'groups': product_data['country']})
    
    # Extract coefficients
    coefs_prod = {tau: results_prod.params.get(f'event_{tau:+d}', 0.0) 
                  for tau in range(-12, 25) if tau != -1}
    
    product_results[product] = coefs_prod

print("\n✓ Product-level event studies complete")

## 9. Compare Effects Across Products

In [ ]:
# Plot comparison
fig, ax = plt.subplots(figsize=(14, 7))

colors = ['navy', 'darkgreen', 'darkred']

for (product, coefs), color in zip(product_results.items(), colors):
    periods = sorted(coefs.keys())
    values = [coefs[t] for t in periods]
    
    ax.plot(periods, values, 'o-', label=product, linewidth=2, markersize=5, color=color, alpha=0.7)

ax.axhline(0, color='black', linestyle='--', linewidth=1.5, alpha=0.5)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.5, label='Tariff')

ax.set_xlabel('Months Relative to Tariff', fontsize=12)
ax.set_ylabel('Coefficient Estimate (Log Points)', fontsize=12)
ax.set_title('Event Study Comparison Across Products', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../viz/event_study_by_product.png', dpi=300, bbox_inches='tight')
plt.show()

## 10. Save Results

In [ ]:
# Save event-study coefficients
event_df.to_csv('../reports/event_study_coefficients.csv', index=False)
print("✓ Saved event-study coefficients to ../reports/event_study_coefficients.csv")

# Save summary statistics
summary = {
    'pre_treatment': {
        'mean_coefficient': float(mean_pre),
        'std_coefficient': float(std_pre),
        'n_significant': int(n_sig_pre),
        'n_periods': int(n_pre)
    },
    'post_treatment': {
        'immediate_effect': float(post_df[post_df['rel_month']==0]['coefficient'].values[0]),
        'mean_0_6mo': float(post_df[post_df['rel_month']<=6]['coefficient'].mean()),
        'mean_7_12mo': float(post_df[(post_df['rel_month']>6) & (post_df['rel_month']<=12)]['coefficient'].mean()),
        'mean_13_24mo': float(post_df[post_df['rel_month']>12]['coefficient'].mean())
    },
    'model': {
        'r_squared': float(results.rsquared),
        'n_obs': int(results.nobs),
        'n_clusters': int(model.data['country'].nunique())
    }
}

import json
with open('../reports/event_study_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("✓ Saved summary statistics to ../reports/event_study_summary.json")

## Summary

### Key Findings:

1. **Pre-treatment trends:** [Describe whether pre-treatment coefficients are close to zero]

2. **Immediate impact:** [Report coefficient at τ=0]

3. **Dynamics:** [Describe how effects evolve over time]

4. **Product heterogeneity:** [Summarize differences across products]

### Interpretation:

The event-study specification allows us to:
- Test for anticipation effects (pre-treatment coefficients)
- Trace out the full dynamic response to the tariff
- Identify heterogeneity across product categories
- Validate the parallel trends assumption visually